# Delta Live Tables

> **Note:** DLT notebooks must be attached to a DLT pipeline to execute — the `dlt` module is only available inside a running pipeline. Code cells in this notebook show the patterns but cannot be run interactively. To test them, create a DLT pipeline in the Databricks UI pointing to this notebook.

**Delta Live Tables (DLT)** is Databricks's declarative pipeline framework. Instead of writing imperative Spark code with explicit orchestration (trigger A, then B, then C), you declare what each table should contain — DLT figures out the execution order, handles retries, tracks data quality, and manages incremental processing automatically.

In this notebook we'll cover:
- The three DLT object types: Live Tables, Streaming Tables, and Views
- Python and SQL syntax for declaring pipeline tables
- The `LIVE.` prefix for referencing pipeline tables
- Expectations — inline data quality rules with configurable failure modes
- `APPLY CHANGES INTO` — automatic CDC and SCD Type 1/2
- Pipeline modes: Development vs Production, Triggered vs Continuous

## Why DLT Exists

A traditional Spark pipeline is a collection of notebooks or scripts chained by a Databricks Job. You are responsible for:
- Writing each transformation step
- Defining the execution order
- Handling retries when a step fails
- Deciding what to recompute vs skip on each run
- Monitoring data quality across tables

DLT eliminates most of this operational overhead:

| Concern | Traditional Pipeline | DLT |
|---------|---------------------|-----|
| Execution order | You define job task dependencies | DLT infers from `LIVE.` references |
| Incremental processing | Manual watermarks, checkpoints | Automatic — DLT tracks what has been processed |
| Retries | Job-level retry config | Automatic per-table retry |
| Data quality | Manual assertions or quarantine | Built-in `EXPECT` constraints with metrics |
| Monitoring | Custom logging | DLT Event Log — unified quality + lineage |
| Schema evolution | Manual `mergeSchema` | Automatic |

## DLT Object Types

### 1. Live Table (Materialized View)

Computed from a batch query. On each pipeline run, DLT recomputes or incrementally refreshes the table. Think of it as a **materialized view** that DLT keeps up to date.

```python
@dlt.table(
    name    = "gold_revenue",
    comment = "Daily revenue by region"
)
def gold_revenue():
    return (
        dlt.read("silver_orders")
           .groupBy("region", "order_date")
           .agg(F.sum("amount").alias("revenue"))
    )
```

SQL equivalent:
```sql
CREATE OR REFRESH LIVE TABLE gold_revenue
COMMENT "Daily revenue by region"
AS
SELECT region, order_date, SUM(amount) AS revenue
FROM   LIVE.silver_orders
GROUP BY region, order_date;
```

### 2. Streaming Table

Backed by a streaming query — processes new data incrementally. Used for Bronze ingestion (Auto Loader) and Silver transformations from a streaming Bronze source.

```python
@dlt.table(
    name    = "bronze_orders",
    comment = "Raw orders from Auto Loader"
)
def bronze_orders():
    return (
        spark.readStream
             .format("cloudFiles")
             .option("cloudFiles.format", "json")
             .option("cloudFiles.schemaLocation", "/dlt/schema/orders")
             .load("s3://bucket/landing/orders/")
    )
```

SQL equivalent:
```sql
CREATE OR REFRESH STREAMING TABLE bronze_orders
AS SELECT * FROM cloud_files(
  's3://bucket/landing/orders/',
  'json',
  map('cloudFiles.schemaLocation', '/dlt/schema/orders')
);
```

### 3. View

Temporary — not materialised to storage. Useful for intermediate transformations you don't want to persist.

```python
@dlt.view(name="validated_orders")
def validated_orders():
    return dlt.read_stream("bronze_orders").filter("amount > 0")
```

| | Live Table | Streaming Table | View |
|---|---|---|---|
| Stored in Delta | Yes | Yes | No |
| Processing | Batch (full or incremental) | Streaming (append-only) | Computed inline |
| Queryable outside pipeline | Yes | Yes | No |
| Use case | Aggregations, Gold layer | Bronze, incremental Silver | Intermediate steps |

## The `LIVE.` Prefix and Dependency Resolution

To read another DLT table within the same pipeline, use the `LIVE.` prefix in SQL or `dlt.read()` / `dlt.read_stream()` in Python:

```sql
-- SQL: reference a live table in the same pipeline
SELECT * FROM LIVE.bronze_orders

-- SQL: reference a streaming live table
SELECT * FROM STREAM(LIVE.bronze_orders)
```

```python
# Python: batch read of a Live Table
dlt.read("bronze_orders")

# Python: streaming read of a Streaming Table
dlt.read_stream("bronze_orders")
```

DLT **automatically builds the dependency graph** from these references. If `silver_orders` reads from `bronze_orders`, DLT ensures `bronze_orders` is up to date before computing `silver_orders` — no explicit task ordering needed.

> **Exam tip:** `LIVE.` is only for tables **within the same pipeline**. To read a table outside the pipeline (e.g., a lookup table in another schema), use a normal table reference — `catalog.schema.table` — without `LIVE.`.

## Expectations — Data Quality Rules

Expectations are inline data quality constraints. They evaluate a boolean SQL expression on every row and take a configured action when the constraint fails.

### Three Failure Modes

```python
@dlt.table
@dlt.expect("valid_amount", "amount > 0")                  # warn — row still written
@dlt.expect_or_drop("non_null_customer", "customer_id IS NOT NULL")  # drop failing rows
@dlt.expect_or_fail("valid_order_id", "LENGTH(order_id) = 6")        # fail the pipeline
def silver_orders():
    return dlt.read_stream("bronze_orders")
```

SQL:
```sql
CREATE OR REFRESH STREAMING TABLE silver_orders
  (
    CONSTRAINT valid_amount      EXPECT (amount > 0),
    CONSTRAINT non_null_customer EXPECT (customer_id IS NOT NULL) ON VIOLATION DROP ROW,
    CONSTRAINT valid_order_id    EXPECT (LENGTH(order_id) = 6)    ON VIOLATION FAIL UPDATE
  )
AS SELECT * FROM STREAM(LIVE.bronze_orders);
```

| Decorator / SQL | Rows that fail | Pipeline | Metrics tracked |
|----------------|---------------|----------|-----------------|
| `expect` / `EXPECT` | Written to table | Continues | Yes — violation count |
| `expect_or_drop` / `ON VIOLATION DROP ROW` | Dropped (not written) | Continues | Yes — dropped count |
| `expect_or_fail` / `ON VIOLATION FAIL UPDATE` | Pipeline fails | Stops | Yes |

### Why Expectations Matter

Expectation metrics are stored in the **DLT Event Log** — queryable as a Delta table. This gives you a historical record of data quality violations over time, enabling dashboards and alerts without custom logging code.

## APPLY CHANGES INTO — Automatic CDC

The most powerful DLT feature for data engineers: `APPLY CHANGES INTO` processes a CDC feed and maintains a Silver table as either SCD Type 1 (overwrite) or SCD Type 2 (full history) — automatically, without writing MERGE logic.

### Python

```python
# Declare the target table first
dlt.create_streaming_table("silver_customers")

# Apply changes from the CDC source
dlt.apply_changes(
    target          = "silver_customers",
    source          = "bronze_cdc",          # streaming source table
    keys            = ["cust_id"],            # primary key columns
    sequence_by     = col("updated_at"),      # ordering column (handles out-of-order)
    apply_as_deletes= expr("op = 'D'"),       # which rows are deletes
    stored_as_scd_type = 1                    # 1 = overwrite, 2 = keep history
)
```

### SQL

```sql
CREATE OR REFRESH STREAMING TABLE silver_customers;

APPLY CHANGES INTO LIVE.silver_customers
FROM   STREAM(LIVE.bronze_cdc)
KEYS   (cust_id)
APPLY AS DELETE WHEN op = 'D'
SEQUENCE BY updated_at
STORED AS SCD TYPE 1;
```

### SCD Type 2

```sql
APPLY CHANGES INTO LIVE.silver_customers_history
FROM   STREAM(LIVE.bronze_cdc)
KEYS   (cust_id)
APPLY AS DELETE WHEN op = 'D'
SEQUENCE BY updated_at
STORED AS SCD TYPE 2;    -- adds __START_AT, __END_AT, __IS_CURRENT columns
```

With SCD Type 2, DLT automatically adds:
- `__START_AT` — when this version became active
- `__END_AT` — when this version was superseded (null for current)
- `__IS_CURRENT` — boolean flag for the active version

**Key properties of `APPLY CHANGES INTO`:**
- Handles **out-of-order records** — `sequence_by` determines which record wins, not arrival order
- Handles **duplicate records** automatically — same key + same sequence value is deduplicated
- Works with **streaming sources only** — the source must be a Streaming Table

> **Exam tip:** `APPLY CHANGES INTO` requires the target to be a **Streaming Table**, not a Live Table. The target table must be declared with `CREATE OR REFRESH STREAMING TABLE` (or `dlt.create_streaming_table`) **before** the `APPLY CHANGES INTO` statement.

## A Complete DLT Pipeline (Bronze → Silver → Gold)

```python
import dlt
from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr

# ─── BRONZE ───────────────────────────────────────────────────────────────────

@dlt.table(
    name    = "bronze_orders",
    comment = "Raw orders from landing zone via Auto Loader"
)
def bronze_orders():
    return (
        spark.readStream
             .format("cloudFiles")
             .option("cloudFiles.format",         "json")
             .option("cloudFiles.schemaLocation", "/dlt/schema/orders")
             .load("s3://bucket/landing/orders/")
             .select(
                 "*",
                 col("_metadata.file_name").alias("_source_file")
             )
    )


# ─── SILVER ───────────────────────────────────────────────────────────────────

@dlt.table(
    name             = "silver_orders",
    comment          = "Cleaned, typed orders",
    table_properties = {"quality": "silver"}
)
@dlt.expect("positive_amount",   "amount > 0")
@dlt.expect_or_drop("valid_customer", "customer_id IS NOT NULL")
def silver_orders():
    return (
        dlt.read_stream("bronze_orders")
           .withColumn("amount",     F.try_cast(col("amount"), "double"))
           .withColumn("order_date", F.to_date(col("order_date")))
           .withColumn("region",     F.trim(col("region")))
    )


# ─── GOLD ─────────────────────────────────────────────────────────────────────

@dlt.table(
    name    = "gold_daily_revenue",
    comment = "Daily revenue by region — aggregated from Silver"
)
def gold_daily_revenue():
    return (
        dlt.read("silver_orders")   # batch read — Gold is a Live Table
           .groupBy("region", "order_date")
           .agg(
               F.sum("amount").alias("revenue"),
               F.count("*").alias("order_count")
           )
    )
```

## Pipeline Configuration

### Development vs Production Mode

| | Development | Production |
|---|---|---|
| Cluster reuse | Cluster persists between runs (faster iteration) | New cluster per run |
| Error handling | Retries disabled — fail fast for debugging | Automatic retries |
| Cost | Higher (cluster stays up) | Lower (cluster only during run) |
| When to use | Building and testing the pipeline | Scheduled production runs |

### Triggered vs Continuous

| | Triggered | Continuous |
|---|---|---|
| Runs | On demand or scheduled | Runs forever |
| Latency | Minutes (depends on schedule) | Seconds |
| Cost | Lower (cluster off between runs) | Higher (cluster always on) |
| When to use | Batch-style pipelines, scheduled Gold updates | Low-latency streaming Bronze/Silver |

### Update Types

| Update Type | Behaviour |
|-------------|----------|
| **Refresh** (incremental) | Processes only new data since the last run — default for streaming tables |
| **Full refresh** | Drops and recomputes all tables from scratch — use after fixing bugs or schema issues |
| **Selective full refresh** | Full refresh of a specific table and its downstream dependencies only |

> After fixing a data quality bug in a Silver table, run a **Full Refresh** to reprocess all Bronze data through the corrected logic. Without it, already-processed records remain uncorrected.

## Target Schema / Catalog

DLT publishes all pipeline tables to a **target schema** (and optionally a catalog) configured in the pipeline settings:

```json
{
  "target": "silver",
  "catalog": "prod_catalog"
}
```

Tables declared in the pipeline appear as `prod_catalog.silver.bronze_orders`, `prod_catalog.silver.silver_orders`, etc. They are fully queryable via Databricks SQL, notebooks, and BI tools once the pipeline has run.

Without a target schema, tables exist only within the pipeline and are not accessible externally.

## Monitoring — The DLT Event Log

DLT writes structured event records to a Delta table called the **event log**, stored at the pipeline's storage location. Query it to inspect pipeline runs, data quality metrics, and table lineage:

```sql
-- Find the event log path from the pipeline settings, then:
SELECT
  timestamp,
  event_type,
  details:flow_name        AS table_name,
  details:expectations     AS quality_metrics
FROM delta.`dbfs:/pipelines/<pipeline-id>/system/events`
WHERE event_type = 'flow_progress'
ORDER BY timestamp DESC
LIMIT 20;
```

Key event types:

| Event type | What it means |
|------------|---------------|
| `flow_progress` | A table (flow) started, ran, or completed |
| `create_update` | A pipeline run was triggered |
| `dataset_definition` | Schema of a declared table |
| `planning_information` | Execution plan — which tables will be computed |

Expectation metrics appear inside `flow_progress` events — you can query violation counts per table per run to build data quality dashboards.

## Key Syntax Comparison — Python vs SQL

```
Goal                          Python                          SQL
──────────────────────────────────────────────────────────────────────────────
Live Table                    @dlt.table                      CREATE OR REFRESH LIVE TABLE
Streaming Table               @dlt.table + readStream         CREATE OR REFRESH STREAMING TABLE
View                          @dlt.view                       CREATE LIVE VIEW
Read live table (batch)       dlt.read("name")                FROM LIVE.name
Read live table (stream)      dlt.read_stream("name")         FROM STREAM(LIVE.name)
Warn on bad data              @dlt.expect("n", "cond")        CONSTRAINT n EXPECT (cond)
Drop bad rows                 @dlt.expect_or_drop(...)        CONSTRAINT n EXPECT (cond) ON VIOLATION DROP ROW
Fail on bad data              @dlt.expect_or_fail(...)        CONSTRAINT n EXPECT (cond) ON VIOLATION FAIL UPDATE
CDC / SCD                     dlt.apply_changes(...)          APPLY CHANGES INTO LIVE.target ...
```

## Summary

- **DLT** is a declarative pipeline framework — you declare tables, DLT infers execution order, handles retries, manages incremental state, and tracks data quality.
- Three object types: **Live Tables** (batch materialized views), **Streaming Tables** (incremental, append-only), **Views** (temporary, not stored).
- Use `LIVE.` prefix in SQL or `dlt.read()` / `dlt.read_stream()` in Python to reference tables within the same pipeline. DLT builds the dependency graph automatically.
- **Expectations** enforce data quality inline: `expect` (warn), `expect_or_drop` (drop rows), `expect_or_fail` (stop pipeline). Metrics land in the Event Log.
- **`APPLY CHANGES INTO`** processes CDC feeds automatically — SCD Type 1 (overwrite) or Type 2 (history with `__START_AT`, `__END_AT`, `__IS_CURRENT`). Target must be a Streaming Table declared before the `APPLY CHANGES INTO`.
- **Development mode** persists the cluster and disables retries — for iterating. **Production mode** uses fresh clusters with retries — for scheduled runs.
- **Triggered** pipelines run on demand/schedule (lower cost). **Continuous** pipelines run forever (lower latency).
- After fixing a bug, use **Full Refresh** to reprocess all data; incremental refresh processes only new data.
- Tables are published to the configured **target schema/catalog** and are queryable like any Delta table.

Next: `13-dlt-expectations-data-quality.ipynb` — deeper dive into expectations, multi-expectation patterns, and querying the Event Log for quality metrics.